# Lab 20-12 · Why Analog Does Not Scale Like Digital

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import scaling


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Scaling factors

*Predict* the gate's energy and the analog stage's power after three nodes of S = 1.4.

In [2]:
print("nodes  delay  energy  area   analog C  analog P")
for k in range(4):
    s = 1.4 ** k
    f = scaling.constant_field(s)
    print(f"{k:5d}  {f['delay']:.3f}  {f['energy']:.3f}  {f['area']:.3f}  {s ** 2:8.2f}  {s:8.2f}")

nodes  delay  energy  area   analog C  analog P
    0  1.000  1.000  1.000      1.00      1.00
    1  0.714  0.364  0.510      1.96      1.40
    2  0.510  0.133  0.260      3.84      1.96
    3  0.364  0.048  0.133      7.53      2.74


## Part 2 · Power against supply

*Predict* how the buffer's power changes from 3.3 V to 0.6 V, first with a proportional swing, then with 0.3 V of fixed headroom.

In [3]:
vdd = np.linspace(0.6, 3.3, 200)
p1 = [scaling.thermal_power(74, 1e8, v)[3] * 1e6 for v in vdd]
p2 = [scaling.thermal_power(74, 1e8, v, eta=(v - 0.3) / v)[3] * 1e6 for v in vdd]
plt.figure()
plt.semilogy(vdd, p1, label="swing 0.8 VDD")
plt.semilogy(vdd, p2, label="swing VDD - 0.3 V")
plt.xlabel("VDD (V)"); plt.ylabel("power (uW)"); plt.legend(); plt.show()
for v in (1.8, 0.8):
    c, gm, i, p = scaling.thermal_power(74, 1e8, v)
    print(f"{v} V: C = {c * 1e12:.3f} pF, I = {i * 1e6:.1f} uA, P = {p * 1e6:.1f} uW")

1.8 V: C = 0.401 pF, I = 16.8 uA, P = 30.3 uW
0.8 V: C = 2.032 pF, I = 85.1 uA, P = 68.1 uW


## Part 3 · The cascode in ngspice

*Predict* the output resistance of each sink at 0.6 V, and of the cascode at 0.3 V.

In [4]:
h = 1e-5
r_single = 2 * h / (scaling.level1_id(0.55, 0.6 + h) - scaling.level1_id(0.55, 0.6 - h))
r_c06 = 2 * h / (scaling.cascode_current(0.6 + h)[0] - scaling.cascode_current(0.6 - h)[0])
r_c03 = 2 * h / (scaling.cascode_current(0.3 + h)[0] - scaling.cascode_current(0.3 - h)[0])
print(f"model:   single {r_single / 1e3:.1f} kohm, cascode {r_c06 / 1e6:.2f} Mohm at 0.6 V, {r_c03 / 1e3:.0f} kohm at 0.3 V")
res = ngspice("stack.cir")
print(f"ngspice: single {res['rsk']:.1f} kohm, cascode {res['rcm']:.2f} Mohm at 0.6 V, {res['rck']:.0f} kohm at 0.3 V")

model:   single 74.1 kohm, cascode 5.58 Mohm at 0.6 V, 232 kohm at 0.3 V
ngspice: single 74.1 kohm, cascode 5.58 Mohm at 0.6 V, 232 kohm at 0.3 V


## Part 4 · Mismatch or noise?

*Predict* the capacitance ratio $C_{ox}A_{VT}^2/kT$ for each process.

In [5]:
kt = 1.380649e-23 * 300
for name, cox, avt in (("Example 20-12.4", 10e-15, 3.5e-3), ("half oxide", 20e-15, 1.75e-3), ("Problem 20-12.8", 25e-15, 1.5e-3)):
    print(f"{name:16s} ratio {cox * avt ** 2 / kt:5.1f}")

Example 20-12.4  ratio  29.6
half oxide       ratio  14.8
Problem 20-12.8  ratio  13.6


## Part 5 · Calibration

*Predict* the SNDR of the two-step converter with a 6.25% residue-gain error, uncorrected and corrected.

In [6]:
n, k = 8192, 1021
x = 0.5 + 0.49 * np.sin(2 * np.pi * k * np.arange(n) / n)
cases = [("no gain error", 0, 0), ("6.25%, uncorrected", 0.0625, 0), ("6.25%, exact correction", 0.0625, 0.0625),
         ("estimate 0.39% off", 0.0625, 0.0625 * (1 - 0.0039)), ("estimate 2% off", 0.0625, 0.0625 * 1.02)]
for name, e, eh in cases:
    print(f"{name:26s} SNDR {scaling.sndr(scaling.two_step_adc(x, e, eh), x):.1f} dB")
err = scaling.two_step_adc(x, 0.0625, 0) - x
plt.figure()
plt.plot(x[:400], err[:400] * 4096, ".", ms=2)
plt.xlabel("input"); plt.ylabel("error (12-bit LSB), uncorrected"); plt.show()

no gain error              SNDR 73.9 dB
6.25%, uncorrected         SNDR 50.1 dB
6.25%, exact correction    SNDR 73.4 dB
estimate 0.39% off         SNDR 73.3 dB
estimate 2% off            SNDR 73.0 dB
